# Pycavi zero point calibration check

This notebook checks the calibration of swarped image calibrated in the previous step.

- Reads the swarped image and some necessary keywords.
- Read the catalogue used to calibrate the image.
- Produce several plots that can be used to determine the quality of the photometry.
- Saves the resulting plot into a file
- Then fit a 2D Chebyshev to the photometry residuals and plot the corrected magnitudes.



In [ ]:
import os
import sys
import glob
import copy
import logging

import numpy as np
from numpy.polynomial import chebyshev
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import cm
from mpl_toolkits.axes_grid1 import make_axes_locatable

from pykrige.ok import OrdinaryKriging  # Ordinary Kriging from pykrige
from pykrige.uk import UniversalKriging  # Ordinary Kriging from pykrige

from astropy.stats import sigma_clipped_stats
from astropy.visualization import simple_norm
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.io import fits
from astropy.wcs import WCS

from photutils.aperture import SkyCircularAperture, SkyCircularAnnulus, aperture_photometry


#   from pycavi_astromatic import SExtractor
sys.path.append(os.path.join(os.path.abspath(''), '..')) # to be removed in production
from pycavi import pycavi_tools
from pycavi.pycavi_zpcal import extract_calibration, plot_zpcal, get_files_dict


In [ ]:
def random_sampling_no_repeats(n, n_sel):
    """
    Creates a series of indices arrays of length n_sel
    randomly selected from n.
    """
    # Create an array of consecutive integers from 0 to n
    arr = np.arange(n+1)

    # List to store all selected indices over iterations
    selected_history = set()

    # List to store the results
    results = []

    # Loop until all indices have been selected at least once
    while len(selected_history) < (n + 1):
        # Determine the indices that are still available to be selected (not in selected_history)
        available_indices = np.setdiff1d(arr, list(selected_history))

        # Randomly select n_sel indices from available indices without replacement
        n_sel_actual = min(n_sel, len(available_indices))  # Adjust in case fewer indices remain
        selected_indices = np.random.choice(available_indices, size=n_sel_actual, replace=False)

        # Add the selected indices to the history set
        selected_history.update(selected_indices)

        # The remaining unselected indices are the complement of the selected ones
        unselected_indices = np.setdiff1d(arr, selected_indices)

        # Append the tuple of selected and unselected arrays
        results.append((selected_indices, unselected_indices))

    return results

# Use a call to uk to determine parameters
def kriging_interpolation(x_subset, y_subset, z_subset, 
                init_kriging = True, 
                selected_model = 'gaussian', 
                maxlag = 12000, 
                nlags = 200, 
                determine_empirical_variogram = True, 
                plot_variogram = False,
                use_empirical_variogram = False):
    """
    Determine universal kriging interpolator using z values on a
    set of points.

    Parameters

        x_subset, y_subset, z_subset:    Values with 2D pixel coordinates
        init_kriging:                    Do an initial run of UniversalKriging with default variogram
        selected_model:                  Which model to use ['gaussian'] 'exponential', 'spherical'
        maxlag:                          Maximum lag to use in fit. Tipically half the size of smallest axis
                                         to avoid edge effects
        nlags:                           Number of bins. Optimal value depends on number of input points.
        determine_empirical_variogram:   Flag to instruct the determination of an empirical variogram
        plot_variogram:                  Instruct to plot the empirical variogram
        use_empirical_variogram:         Flag to instruct to use the empirical variogram to fit the model,
                                         and pass these model parameters to UK.

    Output

        UK:                              An instance of pykrige UniversalKriging object
        
    """
    # set Kriging parameters
    maxlag = 12000
    nlags = 200
    selected_model = 'gaussian'        # exponential, gaussian, or spherical
    init_kriging = True                # Use a call to uk to determine parameters
    
    use_empirical_variogram = False
    determine_empirical_variogram = True
    if determine_empirical_variogram:
        # Combine x and y into a 2D array
        coordinates = np.column_stack((x_subset, y_subset))

        # Step 1: Calculate the pairwise distances between all points
        distances = pdist(coordinates)

        # ERASE
        # Convert to a square distance matrix
        #dist_matrix = squareform(distances)

        # Step 2: Calculate the empirical variogram
        def empirical_variogram(lags, values, bins):
            ''' Calculate empirical variogram for given lags and z-values (values) '''
            semivariance = np.zeros(len(bins) - 1)

            # Calculate the pairwise differences for the z values
            pairwise_diffs = pdist(values.reshape(-1, 1))**2  # Squared differences in z values

            # For each bin, compute the mean semivariance
            for i in range(len(bins) - 1):
                in_bin = np.logical_and(lags >= bins[i], lags < bins[i + 1])
                if np.sum(in_bin) > 0:
                    semivariance[i] = np.mean(pairwise_diffs[in_bin])
            return semivariance

        # Calculate lags and bins
        edists = pdist(coordinates)
        ebins = np.linspace(0, np.max(edists), nlags)
        esemivariance = 0.5*empirical_variogram(edists, z_subset, ebins)
        elags = ebins[:-1]
        print(f"edists.shape: {edists.shape}  elags.shape: {elags.shape}  esemivariance.shape: {esemivariance.shape}")


    # Interpolate onto the grid
    # Here I add new code
    # Define the grid size
    #n_pts = 50
    #gridx = np.linspace(1, zpcal.naxis1, n_pts)
    #gridy = np.linspace(1, zpcal.naxis2, n_pts)

    # Kriging model construction
    if init_kriging:
        UK = UniversalKriging(
            x_subset, y_subset, z_subset,
            variogram_model=selected_model,
            verbose=True,
            nlags=nlags,
            enable_plotting=False
        )

        # the parameters
        fitted_variogram_parameters = UK.variogram_model_parameters
        print(f"Fitted variogram parameters: {fitted_variogram_parameters}\n")

        # the variogram
        uk_lags = UK.lags
        uk_semivariance = UK.semivariance
        uk_model_values = UK.variogram_function(fitted_variogram_parameters, uk_lags)

    # Chose which variogram to use
    if use_empirical_variogram:
        elags = ebins[:-1]
        valid_idx = elags <= maxlag
        lags_truncated = elags[valid_idx]
        semivariance_truncated = esemivariance[valid_idx]
    else:
        valid_idx = uk_lags <= maxlag
        lags_truncated = uk_lags[valid_idx]
        semivariance_truncated = uk_semivariance[valid_idx]
        uk_model_values_truncated = UK.variogram_function(fitted_variogram_parameters, lags_truncated)

    # Step 3: Define the  variogram model functions
    # xponential, gaussian, and spherical variogram model
    #

    def exponential_variogram(h, nugget, partial_sill, range_):
        return nugget + partial_sill * (1 - np.exp(-h / range_))

    def gaussian_variogram(h, nugget, partial_sill, range_):
        return nugget + partial_sill * (1 - np.exp(-h**2 / range_**2))

    def spherical_variogram(h, nugget, partial_sill, range_, ):
        return np.where(
            h <= range_,
            nugget + partial_sill * (1.5 * (h / range_) - 0.5 * (h / range_)**3),
            nugget + partial_sill
        )
    model_function = {'exponential': exponential_variogram,
                      'gaussian': gaussian_variogram,
                      'spherical': spherical_variogram}
    # Select the model
    model = model_function[selected_model]
    
    # Step 4: Fit the model to the empirical variogram using curve_fit
    # Use initial guesses for nugget, sill, and range
    initial_guess = [0.1,0.1, 500.0]

    # Perform the fitting
    bounds=[(0, 0, 100.),(np.inf, np.inf, 5000.)]
    popt, pcov = curve_fit(model, lags_truncated, semivariance_truncated, p0=initial_guess, bounds=bounds)

    # Extract the fitted parameters
    fitted_nugget, fitted_partial_sill, fitted_range = popt
    variogram_parameters = {'psill': fitted_partial_sill, 'range': fitted_range, 'nugget': fitted_nugget}     # NB THE ORDER IS DIFFERENT
    print(variogram_parameters)
    print(f"Fitted Nugget: {fitted_nugget}, Partial Sill: {fitted_partial_sill}, Range: {fitted_range}")

    if plot_variogram:
        plt.figure(figsize=(8, 5))

        # Plot empirical variogram
        plt.scatter(lags_truncated, semivariance_truncated, color='blue', label='Empirical Variogram Truncated')
        plt.scatter(elags, esemivariance, color='green', marker='+', s=5, label='Full Empirical Variogram')

        # Plot model-fitted variogram
        plt.plot(lags_truncated, model_function[selected_model](lags_truncated, *popt), color='red', label='Fitted Variogram Model')
        plt.plot([0], [fitted_nugget], color='gold', marker='o')

        plt.xlabel('Lag Distance')
        plt.ylabel('Semivariance')
        plt.title(f"Empirical Variogram and Fitted Model {selected_model}")
        plt.legend()
        plt.show()

    # Now we run for a second time UK but this time with the selected model and parameters
    UK = UniversalKriging(
        x_subset, y_subset, z_subset,
        variogram_model=selected_model,
        variogram_parameters = variogram_parameters,
        verbose=True,
        enable_plotting=False
    )

    return UK

def determine_kriging_errors(x, y, z, 
                             k=10,
                             init_kriging=True, 
                             selected_model='gaussian', 
                             maxlag=1, 
                             nlags=6, 
                             determine_empirical_variogram=True, 
                             plot_variogram=False, 
                             use_empirical_variogram=True):
    """
    k-fold cross-validation procedure
    
    Does repeated kriging interpolations selecting a subset of A% points for checking errors
    and (100-A)% to carry out the interpolation. The number of runs is equal to 100/A

    Parameters

        x, y, z:          Positions and values of points to determine kriging
        A:                Percent of points to use for checking errors
        kargs:            Same as in procedure kriging_interpolation

    Output

        x_check, y_check, z_check, z_corr: Arrays with same dimensions as x, y, z with the corrected
                                           values in the interpolation.
    """
    # we loop over a whole set of random samplings of points to determine
    # error estimates.
    # Percentage k (e.g., 40% would be k = 40)
    num_check = int(len(zpcal.x_masked) * k / 100)                # Calculate how many elements correspond to k%
    logging.info(f"num_check: {num_check}    len(zpcal.x_masked): {len(zpcal.x_masked)}")

    # we produce sets of n-num_check random indices for interpolations and num_check for error determinations
    # in total sum(num_check) = n
    #
    idx_tuple_arrays = random_sampling_no_repeats(len(zpcal.x_masked)-1, num_check)    # this return a list of pairs of index arrays
    x_check = []
    y_check = []
    z_check = []
    z_corr = []
    for atuple in idx_tuple_arrays:
        logging.info(f"len(atuple[0]): {len(atuple[0])}    len(atuple[1]): {len(atuple[1])}")
        x_check_i = x[atuple[0]]
        y_check_i = y[atuple[0]]
        z_check_i = z[atuple[0]]
        x_subset_i = x[atuple[1]]
        y_subset_i = y[atuple[1]]
        z_subset_i = z[atuple[1]]

        plot_variogram = False
        UK = kriging_interpolation(x_subset_i, y_subset_i, z_subset_i, 
                    init_kriging = init_kriging, 
                    selected_model = selected_model, 
                    maxlag = maxlag, 
                    nlags = nlags, 
                    determine_empirical_variogram = determine_empirical_variogram, 
                    plot_variogram = plot_variogram,
                    use_empirical_variogram = determine_empirical_variogram)
        dpix = 1
        corr, _ = UK.execute('points', x_check_i, y_check_i)
        z_corr_i = z_check_i - corr

        x_check.append(x_check_i)
        y_check.append(y_check_i)
        z_check.append(z_check_i)
        z_corr.append(z_corr_i)
    # Now we have a full set of check points evaluated
    # independently of the Kriging interpolation
    #
    x_check = np.concatenate(x_check)
    y_check = np.concatenate(y_check)
    z_check = np.concatenate(z_check)
    z_corr = np.concatenate(z_corr)

    return x_check, y_check, z_check, z_corr

In [ ]:
# set the directory folders
config_dir = os.getcwd()+'/../config/'
config = pycavi_tools.read_pycavi_config(config_dir)

observatory = config['Provenance']['observatory'] # Needed for the atmospheric extinction curve
project_dir = config['Directories']['project_dir']
reduced_std_dir = config['Directories']['reduced_std_dir']
reduced_science_dir = config['Directories']['reduced_science_dir']  
log_dir = config['Directories']['project_logs_dir']

log_file_swarp = log_dir + "/pymeffpp_zpcal_check.log"
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s:%(levelname)s:%(message)s",
                    handlers=[
                              logging.FileHandler(log_file_swarp),
                              logging.StreamHandler()])
logging.getLogger().setLevel(logging.INFO)

# At this point it is believed that the data reduction process
# should be closely monitored, so we calibrate target/filter sets,
# that is each data partition, one by one.
#
filters = []
reject_observations = []
dir_name = config['Directories']['cal_dir_name']
zpcal_dict = get_files_dict(reduced_std_dir, reduced_science_dir, dir_name, filters, reject_observations)

observation_tags = list(set(zpcal_dict.keys()))
observation_tags.sort()    # Needed to ensure same index in every run
images_to_check = []
for i in range(len(observation_tags)):
    logging.debug("index: {} observation_tag: {}".format(i, observation_tags[i]))
    for j in range(len(zpcal_dict[observation_tags[i]])):
        images_to_check.append(zpcal_dict[observation_tags[i]][j])

for i in range(len(images_to_check)):
    logging.info("index: {} -- file: {}".format(i, images_to_check[i]))



In [ ]:
# Here we loop over the images selected
# See list above to select a data partition to calibrate
# For example, if the list is:
#
# index: 0 -- file: /Users/fselman/Pictures/OCAM/MultiObject/science/CP_Pup/Swarp_dir/CP_Pup_r_SDSS.coadd.fits
# index: 1 -- file: /Users/fselman/Pictures/OCAM/MultiObject/science/M67/Swarp_dir/M67_g_SDSS.coadd.fits
# index: 2 -- file: /Users/fselman/Pictures/OCAM/MultiObject/science/M67/Swarp_dir/M67_r_SDSS.coadd.fits
# index: 3 -- file: /Users/fselman/Pictures/OCAM/MultiObject/science/M67/Swarp_dir/M67_NB_659.coadd.fits
# index: 4 -- file: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/Swarp_dir/SA101_g_SDSS.coadd.fits
# index: 5 -- file: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/Swarp_dir/SA101_r_SDSS.coadd.fits
# index: 6 -- file: /Users/fselman/Pictures/OCAM/MultiObject/std/SA98/Swarp_dir/SA98_NB_659.coadd.fits
#
# then to calibrate M67_NB_659.coadd.fits set
#
# idx_selected = [3]
#
idx_selected = [1]    # Keep empty to check all files
for i in range(len(images_to_check)):
    
    # skip indices not in list idx_selected
    if not (i in idx_selected or not idx_selected):
        continue
        
    zpcal = extract_calibration(images_to_check[i])
    plot_zpcal(zpcal)
    
logging.info("-------Done-------")

In [ ]:
#from numpy.polynomial.chebyshev import Chebyshev
from scipy.interpolate import griddata, Rbf
from scipy.optimize import curve_fit
from scipy.spatial.distance import pdist, squareform
from numpy.polynomial.chebyshev import chebvander2d
from astropy.stats import sigma_clip
from scipy.stats import norm

###################################### 01 FILTER DATA ######################################
# The data is filtered to get obviously bad values and values
# of faint stars which suffer of intrinsic errors which are
# larger than the effect we want to measure.
#
# The call to the filter_data procedure creates the masked arrays
# zpcal.<array>_masked, where array can be any of x, y, z, etc.
#
n_before_filter = len(zpcal.x)
zpcal.filter_data(dm_min=-0.4, dm_max=0.4, brightest_mag=11.5, faintest_mag=15.0)
n_after_filter = len(zpcal.x_masked)
logging.info("Filtering: number of data points {} ---> {}".format(n_before_filter, n_after_filter))

# We create a ds9 regin file for inspection
zpcal.create_ds9_region()

########## 02  PARTITION DATA: MEASURE AND UNCERTAINTY DETERMINATION PARTITIONS ############
#
#.   subset: to be used to determine approximation
#.   check: to be used to check rms of approximation
#
x_subset = zpcal.x_masked
y_subset = zpcal.y_masked
z_subset = zpcal.z_masked

# some statistics
z_mean_subset, z_median_subset, z_std_subset = sigma_clipped_stats(z_subset, sigma=3, maxiters=5)
print("mean: {}  --  median: {}  --  stdev: {}".format(round(z_mean_subset, 4), round(z_median_subset, 4), round(z_std_subset, 4)))


############################ 03 PLOT RESIDUALS AND PDF FOR UNCORRECTED DATA ####################
#
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,5))

ax1.scatter(x_subset, z_subset, s=2, color='red', label=f'subset sigma: {round(z_std_subset, 3)}')

ax1.axhline(z_mean_subset, color='k', linewidth=1, linestyle='-', label='mean: {}'.format(round(z_mean_subset, 3)))
ax1.axhline(z_mean_subset+z_std_subset, color='k', linewidth=1, linestyle='--', label='mean +- {}'.format(round(z_std_subset, 3)))
ax1.axhline(z_mean_subset-z_std_subset, color='k', linewidth=1, linestyle='--')
ax1.set_title("Magnitude residuals", fontsize=16)
ax1.set_xlabel("RA in pixels", fontsize=14)
ax1.set_ylabel("delta mag", fontsize=14)
ax1.tick_params(axis='x', labelsize=12)
ax1.tick_params(axis='y', labelsize=12)
ax1.legend()

z_val = np.linspace(-0.2, 0.2, 100)
p = norm.pdf(z_val, z_mean_subset, z_std_subset)
ax2.hist(zpcal.z_masked, bins=30, density=True, stacked=True, color='green', alpha=0.7)
plt.plot(z_val, p, 'k', linewidth=2)
ax2.set_title("Distribution of dm", fontsize=16)
ax2.set_xlabel("dm [mag]", fontsize=14)
ax2.set_ylabel("P(dm)", fontsize=14)
ax2.tick_params(axis='x', labelsize=12)
ax2.tick_params(axis='y', labelsize=12)

plt.tight_layout()
plt.show()

#################################### 04 FIT RESIDUAL DATA ###############################
#
method = "Kriging"
if "Chebyshev" in method:
    # Here I add new code
    # Define the grid size
    n_pts = 50j
    grid_x, grid_y = np.mgrid[1:zpcal.naxis1:n_pts,1:zpcal.naxis2:n_pts]
    
    # we fit a 2D Chebyshev polynomial of degrees deg
    deg = (6, 6)
    zpcal.fit_chebyshev(deg)   

    # we print the polynomials explicitly
    zpcal.print_chebyshev_polynomials()

    # we display the results of the fit as an image
    # NB: for Chebyshev we fit the whole data set as the
    #     method does not have the issue of exact interpolation
    #     at the input points, and have fewer degrees of freedom.
    #
    #.    inter_type: 'cubic' or 'rb'
    # inter_type = 'cubic'
    inter_type = 'rbf'
    if 'cubic' in inter_type:
        grid_z_cubic = griddata((zpcal.x_masked, zpcal.y_masked), 
                       zpcal.chebyshev_2d_eval(zpcal.x_masked, zpcal.y_masked), 
                      (grid_x, grid_y), method='cubic')
        grid_z_nearest = griddata((zpcal.x_masked, zpcal.y_masked), 
                       zpcal.chebyshev_2d_eval(zpcal.x_masked, zpcal.y_masked), 
                      (grid_x, grid_y), method='nearest')
        grid_z = np.where(np.isnan(grid_z_cubic), grid_z_nearest, grid_z_cubic)
        z_on_grid = grid_z.T
    elif 'rbf' in inter_type:
        rbf = Rbf(zpcal.x_masked, zpcal.y_masked, zpcal.chebyshev_2d_eval(zpcal.x_masked, zpcal.y_masked), function='cubic')
        grid_z_rbf = rbf(grid_x, grid_y)
        z_on_grid = grid_z_rbf.T
    
    
    
    print("z_on_grid.shape: {}".format(z_on_grid.shape))
    fit_image_title = "Chebyshev fit to magnitude residuals"
    save_plot = zpcal.image_path[:-5]+"_cheb_fit_"+f"{deg[0]}_{deg[1]}_"+inter_type+"_01.png"
    res_plot_str = '_cheb_'+f"{deg[0]}_{deg[1]}_"+inter_type+"_"

    # Due to the low number of degrees of freedom for Chebyshev check points are all points
    x_check = x_subset
    y_check = y_subset
    z_check = z_subset
    z_corr = z_subset - zpcal.chebyshev_2d_eval(x_subset, y_subset)

elif "Kriging" in method:                       # Not working yet - kernel dies
    # Perform Universal Kriging interpolation
    # This is a very expensive process. Data needs to be filtered out.
    # O(N^3+MxN^2) where N: sample data points, M: number of grid points
   
    logging.info(f"number of points for krigging: {len(x_subset)}.  number of points for checking: {len(x_check)}")

    # set Kriging parameters
    maxlag = 12000
    nlags = 200
    selected_model = 'spherical'        # exponential, gaussian, or spherical
    init_kriging = True                # Use a call to uk to determine parameters    
    use_empirical_variogram = True
    determine_empirical_variogram = True

    plot_variogram = True
    UK = kriging_interpolation(x_subset, y_subset, z_subset, 
                    init_kriging = init_kriging, 
                    selected_model = selected_model, 
                    maxlag = maxlag, 
                    nlags = nlags, 
                    determine_empirical_variogram = determine_empirical_variogram, 
                    plot_variogram = plot_variogram,
                    use_empirical_variogram = determine_empirical_variogram)

    # We do 100/A runs to determine kriging errors
    A = 10      # Percentual number of points to use for checking interpolation
    determine_errors = True
    if determine_errors is True:
        x_check, y_check, z_check, z_corr = determine_kriging_errors(x_subset, y_subset, z_subset, 
                                         A=10,
                                         init_kriging=init_kriging, 
                                         selected_model=selected_model, 
                                         maxlag=maxlag, 
                                         nlags=nlags, 
                                         determine_empirical_variogram=determine_empirical_variogram, 
                                         plot_variogram=plot_variogram, 
                                         use_empirical_variogram=use_empirical_variogram)
        print("z_corr.shape: {}".format(z_corr.shape))


    # Interpolate onto the grid
    # Here I add new code
    # Define the grid size
    n_pts = 50
    gridx = np.linspace(1, zpcal.naxis1, n_pts)
    gridy = np.linspace(1, zpcal.naxis2, n_pts)

    # we get the values and the variances
    z_on_grid, ss = UK.execute('grid', gridx, gridy)

    # we create an image
    grid_z = z_on_grid
    grid_x, grid_y = np.meshgrid(gridx, gridy)
    print("z_on_grid.shape: {} --  grid_x.shape: {}".format(z_on_grid.shape, grid_x.shape))
    
    fit_image_title = "Kriging interpolation to magnitude residuals"
    save_plot = zpcal.image_path[:-5]+"_krige_"+selected_model+"_fit_01.png"
    res_plot_str = '_krig_'+selected_model+"_"

logging.info("######################### DONE ######################")


In [ ]:
########################################## 05 PLOT APPROXIMATING SURFACE ###########################
#
fig, ax = plt.subplots(figsize=(12,12))
img = ax.imshow(z_on_grid, extent=(1, zpcal.naxis1, 1, zpcal.naxis2), origin='lower', cmap='viridis')
ax.set_title(fit_image_title, fontsize=24)

divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size="5%", pad=0.1)
cbar = plt.colorbar(img, cax=cax)
cbar.set_label('Residual [mag]', fontsize=14)
cbar.ax.tick_params(labelsize=14)

# Define contours and label them
contour_levels_neg = np.arange(-0.20, 0.0, 0.02)
contour_levels_pos = np.arange(0.02, 0.22, 0.02)

contour_neg = ax.contour(grid_x, grid_y, grid_z, cmap='viridis', levels = contour_levels_neg)
contour_pos = ax.contour(grid_x, grid_y, grid_z, cmap='plasma', levels = contour_levels_pos)
zero_contour = ax.contour(grid_x, grid_y, grid_z, colors='white', linestyles='--', linewidths=2, levels = [0])

ax.clabel(contour_neg, inline=True, fontsize=10, fmt='%1.3f')
ax.clabel(contour_pos, inline=True, fontsize=10, fmt='%1.3f')
ax.clabel(zero_contour, inline=True, fontsize=10, fmt='%1.3f')

# Plot the sample of points used to determine the 2D approximation
XY_subset = np.array(list(zip(x_subset, y_subset)))
ax.scatter(XY_subset[:,0], XY_subset[:,1], s=2, color='white')

# Plot the set of points to be used to check uncertainty
XY_check = np.array(list(zip(x_check, y_check)))
ax.scatter(XY_check[:,0], XY_check[:,1], s=30, color='white', marker='+')


#plt.colorbar(contour, label='Illumination correction')
ax.set_xlabel('RA in pixels', fontsize=16)
ax.set_ylabel('DEC in pixels', fontsize=16)
ax.tick_params(axis='x', labelsize=14)
ax.tick_params(axis='y', labelsize=14)


logging.info("Saving chebyshev fit plot to {}".format(save_plot))
plt.savefig(save_plot)

plt.show()

In [ ]:
######################### PLOT RESIDUALS AND RMS ESTIMATE USING INDEPENDENT DATA POINTS ##################
#

from scipy.stats import norm

#clipped_corr = np.clip(z_cheb_corr, min_corr, max_corr)


z_mean_u, z_median_u, z_std_u = sigma_clipped_stats(z_subset, sigma=3.0, maxiters=5)
print("mean: {}  --  median: {}  --  stdev: {}".format(round(z_mean_u, 4), round(z_median_u, 4), round(z_std_u, 4)))

z_mean, z_median, z_std = sigma_clipped_stats(z_corr, sigma=3, maxiters=5)
print("mean: {}  --  median: {}  --  stdev: {}".format(round(z_mean, 4), round(z_median, 4), round(z_std, 4)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,5))

ax1.scatter(x_subset, z_subset, s=4, color='red', label="mean: {}   stdev: {}".format(round(z_mean_u, 3), round(z_std_u, 3)))
ax1.scatter(x_check, z_corr, color='black', s=15, marker='+', label='mean: {}  stdev: {}'.format(round(z_mean, 3), round(z_std, 3)))
ax1.axhline(z_mean, color='k', linewidth=1, linestyle='-')
ax1.axhline(z_mean+z_std, color='k', linewidth=1, linestyle='--')
ax1.axhline(z_mean-z_std, color='k', linewidth=1, linestyle='--')
ax1.set_title("Magnitude residuals", fontsize=16)
ax1.set_xlabel("RA in pixels", fontsize=14)
ax1.set_ylabel("delta mag", fontsize=14)
ax1.set_ylim(-0.25, 0.25)
ax1.tick_params(axis='x', labelsize=12)
ax1.tick_params(axis='y', labelsize=12)
ax1.legend()

x_val = np.linspace(-0.25, 0.25, 100)
p = norm.pdf(x_val, z_mean, z_std)
ax2.hist(zpcal.z_masked, bins=30, density=True, stacked=True, color='green', alpha=1.0)
ax2.hist(z_corr, bins=30, density=True, stacked=True, color='blue', alpha=0.7)
plt.plot(x_val, p, 'k', linewidth=2)
ax2.set_title("Distribution of dm", fontsize=16)
ax2.set_xlabel("dm", fontsize=14)
ax2.set_ylabel("P(dm)", fontsize=14)
ax2.tick_params(axis='x', labelsize=12)
ax2.tick_params(axis='y', labelsize=12)
ax2.set_ylim(0, 35)

plt.tight_layout()
logging.info("Saving residual plots to {}".format(zpcal.image_path[:-5]+res_plot_str+"res_01.png"))
plt.savefig(zpcal.image_path[:-5]+res_plot_str+"res_01.png")

plt.show()

In [ ]:
############################ EXTENSIVE SPATIAL PLOTS OF RESIDUALS ###########################
#
r = np.sqrt(x_subset**2+y_subset**2)
theta = np.degrees(np.arctan2(y_subset-zpcal.naxis2/2.0, x_subset-zpcal.naxis1/2.0))

r_check = np.sqrt(x_check**2+y_check**2)
theta_check = np.degrees(np.arctan2(y_check-zpcal.naxis2/2.0, x_check-zpcal.naxis1/2.0))

# fit a line to the radial dependence
from scipy import stats

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12,5))
fig.suptitle("Residuals of corrected and uncorrected magnitude differences")

slope, intercept, r_value, p_value, std_err = stats.linregress(r, z_subset)
ax1.scatter(r, z_subset, color='red', s=6, label="uncorrected mags")
xx = np.linspace(0, max(r), 100)
#ax1.plot(xx, slope*xx+intercept, label=f'Fitted line: y = {slope:.3f}x + {intercept:.3f}', color='red')
ax1.plot(xx, slope*xx+intercept, color='red')
ax1.set_xlabel("r [pix]")
ax1.set_ylabel("dm")
ax1.set_ylim(-0.25, 0.25)
               
ax1.scatter(r_check, z_corr, color='black', s=4, label="corrected mags")
slope, intercept, r_value, p_value, std_err = stats.linregress(r_check, z_corr)
xx = np.linspace(0, 1.2, 10)
#ax1.plot(xx, slope*xx+intercept, label=f'Fitted line: y = {slope:.3f}x + {intercept:.3f}', color='black')
ax1.plot(xx, slope*xx+intercept, color='black')

ax1.axhline(0.0, color='k', linewidth=1, linestyle='--')
ax1.legend()

ax2.scatter(theta, z_subset, color='red', s=6, label="Residual of uncorrected mags")
ax2.scatter(theta_check, z_corr, color='black', s=4, label="Residual of uncorrected mags")
ax2.axhline(0.0, color='k', linewidth=1, linestyle='--')
ax2.set_xlabel("theta [deg]")
ax2.set_ylabel("dm")
ax2.set_ylim(-0.25, 0.25)

ax3.scatter(x_subset, z_subset, color='red', s=6, label="Residual of uncorrected mags")
ax3.scatter(x_check, z_corr, color='black', s=2, label="Residual of corrected mags")
ax3.axhline(0.0, color='k', linewidth=1, linestyle='--')
ax3.set_xlabel("RA [pix]")
ax3.set_ylabel("dm")
ax3.set_ylim(-0.25, 0.25)

ax4.scatter(y_subset, z_subset, color='red', s=6, label="Residual of uncorrected mags")
ax4.scatter(y_check, z_corr, color='black', s=2, label="Residual of ucorrected mags")
ax4.axhline(0.0, color='k', linewidth=1, linestyle='--')
ax4.set_xlabel("Dec [pix]")
ax4.set_ylabel("dm")
ax4.set_ylim(-0.25, 0.25)

plt.tight_layout()

logging.info("Saving residual plots to {}".format(zpcal.image_path[:-5]+res_plot_str+"res_02.png"))
plt.savefig(zpcal.image_path[:-5]+res_plot_str+"res_02.png")
               
plt.show()

In [ ]:
# We anlyze the bright saturated end to see if we can model it
# DOES NOT WORK. RESULT IS CLOSER TO STRAIGHT LINE UNTIL
# SAT MAG WHERE IT IS 0.0
#
def sat_model(x, m, c, k, x0):
    return c + (m / k) * np.log(1 + np.exp(k * (x0 -  x)**2))

if "Chebyshev" in method:
    z_std = zpcal.mag_std
    z_frame = zpcal.mag_frame
    dm = z_std - z_frame

    x_pix = zpcal.x
    y_pix = zpcal.y

    z_corr = z_std - (z_frame + zpcal.chebyshev_2d_eval(x_pix, y_pix))

    # Estimación inicial de los parámetros [m, c, k, x0]
    initial_guess = [0.1, 3.0, 2, 10]

    # Ajustar la función a los datos
    good_data = ~np.isnan(z_corr) & np.isfinite(z_corr) & ~np.isnan(z_std) & np.isfinite(z_std)
    params, covariance = curve_fit(sat_model, z_std[good_data], z_corr[good_data], p0=initial_guess)

    # Generar los datos ajustados
    x_data = np.linspace(8, 19, 100)
    y_fitted = sat_model(x_data, *params)

    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    ax.scatter(z_std[good_data], z_corr[good_data], label='Datos', color='blue', s=3)
    ax.plot(x_data, y_fitted, label='Función ajustada', color='red')
    ax.set_ylim(-0.2, 0.2)
    ax.axhline(0.0, color='k', linewidth=1, linestyle='--')

    plt.show()

### 